<a href="https://colab.research.google.com/github/JONATHANPALACIOSParametric/JONATHANPALACIOSParametric/blob/main/Building-Design-Notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install --upgrade openseespy pandas matplotlib plotly -q
print("✓ Instalación de dependencias completada. Ya puedes ejecutar la siguiente celda.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 1.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.3/80.3 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 49.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 60.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 59.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.3/86.3 MB 8.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.0.6 which is incompatible.
✓ Instalación de dependencias completada. Ya puedes ejecutar la siguiente celda.


In [2]:
# ================================================================
# OPENSEESPY 3D STRUCTURAL ANALYSIS & DESIGN
# Parametric Steel Building - Robust Execution Version
# ================================================================

import sys
import traceback

try:
    print("1. Iniciando entorno e importación de librerías...")
    import openseespy.opensees as ops
    import numpy as np
    import pandas as pd
    import plotly.graph_objects as go
    from IPython.display import display

    # Asegurar limpieza absoluta de modelos previos
    ops.wipe()
    print("✓ Entorno de OpenSees inicializado y limpio.")

    # ================================================================
    # 1. MODEL INPUTS
    # ================================================================
    print("2. Definiendo parámetros de entrada (Geometría, Materiales, Secciones)...")
    geometry = {
        "n_bays_x": 4,
        "n_bays_y": 3,
        "n_storeys": 4,
        "bay_x": 6.0,       # m
        "bay_y": 5.0,       # m
        "storey_h": 3.5     # m
    }

    materials = pd.DataFrame([
        {
            "mat_id": 1,
            "name": "S355",
            "E": 210000.0,     # MPa
            "G": 80769.0,      # MPa
            "fy": 355.0        # MPa
        }
    ])

    sections = pd.DataFrame([
        {
            "sec_id": 1,
            "name": "HEB 260",
            "type": "COLUMN",
            "A": 11840.0,       # mm2
            "Iy": 51.34e6,    # mm4
            "Iz": 149.2e6,    # mm4
            "J": 1.238e6,     # mm4
            "Wely": 395e3,    # mm3
            "Welz": 1150e3,   # mm3
            "fy": 355.0
        },
        {
            "sec_id": 2,
            "name": "IPE 300",
            "type": "BEAM",
            "A": 5380.0,
            "Iy": 6.04e6,
            "Iz": 83.56e6,
            "J": 0.201e6,
            "Wely": 80.5e3,
            "Welz": 557e3,
            "fy": 355.0
        }
    ])

    # ================================================================
    # 2. GENERATE NODE TABLE
    # ================================================================
    print("3. Generando coordenadas de nodos...")
    nodes = []
    node_map = {}
    node_id = 1

    for k in range(geometry["n_storeys"] + 1):
        z = float(k * geometry["storey_h"])
        for j in range(geometry["n_bays_y"] + 1):
            y = float(j * geometry["bay_y"])
            for i in range(geometry["n_bays_x"] + 1):
                x = float(i * geometry["bay_x"])
                nodes.append({
                    "Node": node_id,
                    "X": x,
                    "Y": y,
                    "Z": z
                })
                node_map[(i, j, k)] = node_id
                node_id += 1

    df_nodes = pd.DataFrame(nodes)
    print(f"✓ Se generaron {len(df_nodes)} nodos.")

    # ================================================================
    # 3. GENERATE MEMBER TABLE
    # ================================================================
    print("4. Creando topología de miembros estructurales...")
    members = []
    member_id = 1

    # Columns
    for k in range(geometry["n_storeys"]):
        for j in range(geometry["n_bays_y"] + 1):
            for i in range(geometry["n_bays_x"] + 1):
                members.append({
                    "Member": member_id,
                    "NodeI": node_map[(i, j, k)],
                    "NodeJ": node_map[(i, j, k + 1)],
                    "Type": "COLUMN",
                    "Section": 1
                })
                member_id += 1

    # Beams X
    for k in range(1, geometry["n_storeys"] + 1):
        for j in range(geometry["n_bays_y"] + 1):
            for i in range(geometry["n_bays_x"]):
                members.append({
                    "Member": member_id,
                    "NodeI": node_map[(i, j, k)],
                    "NodeJ": node_map[(i + 1, j, k)],
                    "Type": "BEAM",
                    "Section": 2
                })
                member_id += 1

    # Beams Y
    for k in range(1, geometry["n_storeys"] + 1):
        for j in range(geometry["n_bays_y"]):
            for i in range(geometry["n_bays_x"] + 1):
                members.append({
                    "Member": member_id,
                    "NodeI": node_map[(i, j, k)],
                    "NodeJ": node_map[(i, j + 1, k)],
                    "Type": "BEAM",
                    "Section": 2
                })
                member_id += 1

    df_members = pd.DataFrame(members)
    print(f"✓ Se generaron {len(df_members)} miembros estructurales.")

    # ================================================================
    # 4. SUPPORTS
    # ================================================================
    df_supports = pd.DataFrame([
        {
            "Node": int(node["Node"]),
            "UX": 1, "UY": 1, "UZ": 1,
            "RX": 1, "RY": 1, "RZ": 1
        }
        for _, node in df_nodes[df_nodes["Z"] == 0.0].iterrows()
    ])

    # ================================================================
    # 5. LOADS
    # ================================================================
    df_loads = pd.DataFrame([
        {
            "Node": int(node["Node"]),
            "Fx": 0.0, "Fy": 0.0, "Fz": -50.0
        }
        for _, node in df_nodes[df_nodes["Z"] > 0.0].iterrows()
    ])

    # ================================================================
    # 6. BUILD OPENSEES MODEL
    # ================================================================
    print("5. Configurando modelo matemático en el motor OpenSees...")
    ops.model("basic", "-ndm", 3, "-ndf", 6)

    # Cargar nodos
    for _, row in df_nodes.iterrows():
        ops.node(int(row["Node"]), float(row["X"]), float(row["Y"]), float(row["Z"]))

    # Aplicar restricciones de soporte
    for _, row in df_supports.iterrows():
        ops.fix(int(row["Node"]), 1, 1, 1, 1, 1, 1)

    # Transformaciones geométricas robustas para vigas y columnas 3D
    ops.geomTransf("Linear", 1, 0.0, 1.0, 0.0) # Columnas (eje longitudinal Z)
    ops.geomTransf("Linear", 2, 0.0, 0.0, 1.0) # Vigas en X
    ops.geomTransf("Linear", 3, 0.0, 0.0, 1.0) # Vigas en Y

    # Agregar elementos estructurales
    for _, member in df_members.iterrows():
        sec = sections[sections["sec_id"] == member["Section"]].iloc[0]

        # Escalar de mm a metros
        A = float(sec["A"]) * 1e-6
        Iy = float(sec["Iy"]) * 1e-12
        Iz = float(sec["Iz"]) * 1e-12
        J = float(sec["J"]) * 1e-12

        E = float(materials.iloc[0]["E"]) * 1e6
        G = float(materials.iloc[0]["G"]) * 1e6

        # Definir la transformación geométrica según corresponda
        if member["Type"] == "COLUMN":
            transf = 1
        else:
            # Determinar si la viga es paralela a X o a Y
            node_i = df_nodes[df_nodes["Node"] == member["NodeI"]].iloc[0]
            node_j = df_nodes[df_nodes["Node"] == member["NodeJ"]].iloc[0]
            if abs(node_i["Y"] - node_j["Y"]) < 1e-3:
                transf = 2
            else:
                transf = 3

        ops.element(
            "elasticBeamColumn",
            int(member["Member"]),
            int(member["NodeI"]),
            int(member["NodeJ"]),
            A, E, G, J, Iy, Iz, transf
        )
    print("✓ Geometría y elementos de OpenSees cargados exitosamente.")

    # ================================================================
    # 7. LOADS AS PATTERN
    # ================================================================
    ops.timeSeries("Linear", 1)
    ops.pattern("Plain", 1, 1)
    for _, load in df_loads.iterrows():
        ops.load(int(load["Node"]), float(load["Fx"]), float(load["Fy"]), float(load["Fz"]), 0.0, 0.0, 0.0)

    # ================================================================
    # 8. RUN STATIC ANALYSIS
    # ================================================================
    print("6. Ejecutando análisis numérico estático...")
    ops.system("BandGeneral")
    ops.numberer("RCM")
    ops.constraints("Plain")
    ops.integrator("LoadControl", 1.0)
    ops.algorithm("Linear")
    ops.analysis("Static")

    ok = ops.analyze(1)
    if ok == 0:
        print("✓ ¡Análisis estático de OpenSees completado con éxito!")
    else:
        raise RuntimeError("La resolución del sistema de ecuaciones falló en OpenSees.")

    # ================================================================
    # 9. PROCESS NODE DISPLACEMENTS
    # ================================================================
    node_results = []
    for _, node in df_nodes.iterrows():
        tag = int(node["Node"])
        disp = ops.nodeDisp(tag)
        node_results.append({
            "Node": tag,
            "X": node["X"], "Y": node["Y"], "Z": node["Z"],
            "UX [mm]": disp[0] * 1000,
            "UY [mm]": disp[1] * 1000,
            "UZ [mm]": disp[2] * 1000,
            "RX": disp[3], "RY": disp[4], "RZ": disp[5]
        })
    df_node_results = pd.DataFrame(node_results)

    # ================================================================
    # 10. PROCESS INTERNAL MEMBER FORCES
    # ================================================================
    member_results = []
    for _, member in df_members.iterrows():
        tag = int(member["Member"])
        force = ops.eleResponse(tag, "localForce")
        if force is None or len(force) < 12:
            continue

        member_results.append({
            "Member": tag,
            "Type": member["Type"],
            "Section": member["Section"],
            "N_i [kN]": force[0] / 1000, "Vy_i [kN]": force[1] / 1000, "Vz_i [kN]": force[2] / 1000,
            "Mx_i [kNm]": force[3] / 1000, "My_i [kNm]": force[4] / 1000, "Mz_i [kNm]": force[5] / 1000,
            "N_j [kN]": force[6] / 1000, "Vy_j [kN]": force[7] / 1000, "Vz_j [kN]": force[8] / 1000,
            "Mx_j [kNm]": force[9] / 1000, "My_j [kNm]": force[10] / 1000, "Mz_j [kNm]": force[11] / 1000
        })
    df_member_results = pd.DataFrame(member_results)

    # ================================================================
    # 11. STEEL DESIGN CHECK
    # ================================================================
    design_results = []
    for _, result in df_member_results.iterrows():
        member = df_members[df_members["Member"] == result["Member"]].iloc[0]
        sec = sections[sections["sec_id"] == member["Section"]].iloc[0]

        NEd = max(abs(result["N_i [kN]"]), abs(result["N_j [kN]"]))
        MyEd = max(abs(result["My_i [kNm]"]), abs(result["My_j [kNm]"]))
        MzEd = max(abs(result["Mz_i [kNm]"]), abs(result["Mz_j [kNm]"]))

        fy = sec["fy"]
        NplRd = float(sec["A"]) * fy / 1000
        MyRd = float(sec["Wely"]) * fy / 1e6
        MzRd = float(sec["Welz"]) * fy / 1e6

        dcr = max(NEd / NplRd, MyEd / MyRd, MzEd / MzRd)
        design_results.append({
            "Member": result["Member"],
            "Section": sec["name"],
            "NEd [kN]": NEd, "MyEd [kNm]": MyEd, "MzEd [kNm]": MzEd,
            "N_Rd [kN]": NplRd, "My_Rd [kNm]": MyRd, "Mz_Rd [kNm]": MzRd,
            "DCR": dcr,
            "Status": "PASS" if dcr <= 1.0 else "FAIL"
        })
    df_design = pd.DataFrame(design_results)

    # ================================================================
    # 12. DISPLAY TABULAR RESULTS
    # ================================================================
    print("\n=== DESPLAZAMIENTOS NODALES MAXYMOS ===")
    display(df_node_results.style.format(precision=2).background_gradient(subset=["UX [mm]", "UY [mm]", "UZ [mm]"]))

    print("\n=== FUERZAS EN LOS MIEMBROS ===")
    display(df_member_results.head(10).style.format(precision=2))

    print("\n=== VERIFICACIÓN DE DISEÑO DE ACERO ===")
    display(df_design.sort_values("DCR", ascending=False).style.format(precision=2).background_gradient(subset=["DCR"]))

    # ================================================================
    # 13. PLOTLY 3D MODEL VISUALIZATION
    # ================================================================
    fig = go.Figure()
    for _, member in df_members.iterrows():
        ni = df_nodes[df_nodes["Node"] == member["NodeI"]].iloc[0]
        nj = df_nodes[df_nodes["Node"] == member["NodeJ"]].iloc[0]
        fig.add_trace(go.Scatter3d(
            x=[ni.X, nj.X], y=[ni.Y, nj.Y], z=[ni.Z, nj.Z],
            mode="lines", line=dict(width=4, color="blue"), showlegend=False
        ))

    fig.update_layout(
        title="Estructura Paramétrica en 3D",
        scene=dict(xaxis_title="X [m]", yaxis_title="Y [m]", zaxis_title="Z [m]", aspectmode="data"),
        height=600
    )
    fig.show()

    # ================================================================
    # 14. STRUCTURAL DESIGN SUMMARY REPORT
    # ================================================================
    print("\n" + "="*60)
    print("STRUCTURAL DESIGN SUMMARY REPORT")
    print("="*60)
    print(f"Nodos del modelo:       {len(df_nodes)}")
    print(f"Miembros cargados:      {len(df_members)}")
    print(f"Número de Niveles:      {geometry['n_storeys']}")
    print(f"Deflexión Máxima Z:     {df_node_results['UZ [mm]'].abs().max():.4f} mm")
    print(f"Ratio de Trabajo Max:   {df_design['DCR'].max():.4f}")
    print(f"Miembros Aptos (PASS):  {(df_design['Status'] == 'PASS').sum()}")
    print(f"Miembros Fallidos:      {(df_design['Status'] == 'FAIL').sum()}")

except Exception as e:
    print("✗ Ocurrió un error inesperado durante el análisis:")
    traceback.print_exc()


1. Iniciando entorno e importación de librerías...
✓ Entorno de OpenSees inicializado y limpio.
2. Definiendo parámetros de entrada (Geometría, Materiales, Secciones)...
3. Generando coordenadas de nodos...
✓ Se generaron 100 nodos.
4. Creando topología de miembros estructurales...
✓ Se generaron 204 miembros estructurales.
5. Configurando modelo matemático en el motor OpenSees...
✓ Geometría y elementos de OpenSees cargados exitosamente.
6. Ejecutando análisis numérico estático...
✓ ¡Análisis estático de OpenSees completado con éxito!

=== DESPLAZAMIENTOS NODALES MAXYMOS ===


,Node,X,Y,Z,UX [mm],UY [mm],UZ [mm],RX,RY,RZ
0,1,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
1,2,6.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
2,3,12.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
3,4,18.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
4,5,24.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
5,6,0.00,5.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
6,7,6.00,5.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
7,8,12.00,5.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
8,9,18.00,5.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
9,10,24.00,5.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00



=== FUERZAS EN LOS MIEMBROS ===


,Member,Type,Section,N_i [kN],Vy_i [kN],Vz_i [kN],Mx_i [kNm],My_i [kNm],Mz_i [kNm],N_j [kN],Vy_j [kN],Vz_j [kN],Mx_j [kNm],My_j [kNm],Mz_j [kNm]
0,1,COLUMN,1,0.20,0.00,0.00,0.00,0.00,0.00,-0.20,-0.00,-0.00,-0.00,-0.00,-0.00
1,2,COLUMN,1,0.20,0.00,-0.00,0.00,0.00,0.00,-0.20,-0.00,0.00,-0.00,0.00,-0.00
2,3,COLUMN,1,0.20,0.00,-0.00,0.00,0.00,0.00,-0.20,-0.00,0.00,-0.00,0.00,-0.00
3,4,COLUMN,1,0.20,0.00,-0.00,0.00,-0.00,0.00,-0.20,-0.00,0.00,-0.00,0.00,-0.00
4,5,COLUMN,1,0.20,0.00,-0.00,0.00,-0.00,0.00,-0.20,-0.00,0.00,-0.00,0.00,0.00
5,6,COLUMN,1,0.20,0.00,0.00,0.00,0.00,0.00,-0.20,-0.00,-0.00,-0.00,-0.00,-0.00
6,7,COLUMN,1,0.20,0.00,-0.00,0.00,0.00,0.00,-0.20,-0.00,0.00,-0.00,0.00,-0.00
7,8,COLUMN,1,0.20,0.00,-0.00,0.00,0.00,0.00,-0.20,-0.00,0.00,-0.00,0.00,0.00
8,9,COLUMN,1,0.20,0.00,0.00,0.00,-0.00,0.00,-0.20,-0.00,-0.00,-0.00,-0.00,0.00
9,10,COLUMN,1,0.20,0.00,-0.00,0.00,-0.00,0.00,-0.20,-0.00,0.00,-0.00,0.00,0.00



=== VERIFICACIÓN DE DISEÑO DE ACERO ===


,Member,Section,NEd [kN],MyEd [kNm],MzEd [kNm],N_Rd [kN],My_Rd [kNm],Mz_Rd [kNm],DCR,Status
4,5,HEB 260,0.20,0.00,0.00,4203.20,140.22,408.25,0.00,PASS
15,16,HEB 260,0.20,0.00,0.00,4203.20,140.22,408.25,0.00,PASS
16,17,HEB 260,0.20,0.00,0.00,4203.20,140.22,408.25,0.00,PASS
2,3,HEB 260,0.20,0.00,0.00,4203.20,140.22,408.25,0.00,PASS
9,10,HEB 260,0.20,0.00,0.00,4203.20,140.22,408.25,0.00,PASS
1,2,HEB 260,0.20,0.00,0.00,4203.20,140.22,408.25,0.00,PASS
10,11,HEB 260,0.20,0.00,0.00,4203.20,140.22,408.25,0.00,PASS
5,6,HEB 260,0.20,0.00,0.00,4203.20,140.22,408.25,0.00,PASS
19,20,HEB 260,0.20,0.00,0.00,4203.20,140.22,408.25,0.00,PASS
14,15,HEB 260,0.20,0.00,0.00,4203.20,140.22,408.25,0.00,PASS



STRUCTURAL DESIGN SUMMARY REPORT
Nodos del modelo:       100
Miembros cargados:      204
Número de Niveles:      4
Deflexión Máxima Z:     0.0007 mm
Ratio de Trabajo Max:   0.0000
Miembros Aptos (PASS):  204
Miembros Fallidos:      0


In [3]:
# ============================================================
# ATU STRUCTURE - MODEL INPUTS
# ============================================================
# Edit this cell only when changing the structural model.
#
# Units:
#   Geometry       = m
#   Loads          = kN, kN/m²
#   Material E     = MPa
#   Strength       = MPa
#   Reinforcement  = mm
#
# The tables below are used by the OpenSees model.
# ============================================================

import pandas as pd
import numpy as np


# ------------------------------------------------------------
# 1. GENERAL MODEL SETTINGS
# ------------------------------------------------------------

MODEL = {
    "name": "ATU 3D RC Structure",
    "ndm": 3,
    "ndf": 6,

    # Number of storeys represented in the structural model
    "n_storeys": 8,

    # Typical storey height
    "storey_height": 3.50,

    # Analysis type
    "analysis_type": "Linear Elastic",

    # Optional deformation scale for plots
    "deformation_scale": 50,
}


# ------------------------------------------------------------
# 2. GRID / GEOMETRY
# ------------------------------------------------------------
# Based on the structural arrangement:
#
#        A     B     C     D     E    F'     G     H     I
#
# Overall plan approximately 66 m x 28 m.
#
# Change the values below rather than changing the OpenSees
# model itself.
# ------------------------------------------------------------

grid_x = pd.DataFrame({
    "Grid": ["A", "B", "C", "D", "E", "F", "F'", "G", "H", "I"],
    "X_m": [
        0.000,
        7.950,
        16.200,
        24.450,
        32.700,
        40.316,
        41.124,
        49.200,
        57.450,
        65.400
    ]
})


grid_y = pd.DataFrame({
    "Grid": ["1", "2", "3", "4", "5", "6"],
    "Y_m": [
        0.000,
        9.100,
        18.400,
        27.400,
        40.400,
        53.400
    ]
})


# ------------------------------------------------------------
# 3. STOREY LEVELS
# ------------------------------------------------------------
# Change storey height here if required.
# The basement can be added later as additional levels.

storeys = pd.DataFrame({
    "Level": range(0, MODEL["n_storeys"] + 1),
    "Z_m": [
        i * MODEL["storey_height"]
        for i in range(MODEL["n_storeys"] + 1)
    ]
})

print("GRID X")
display(grid_x)

print("\nGRID Y")
display(grid_y)

print("\nSTOREY LEVELS")
display(storeys)


# ------------------------------------------------------------
# 4. MATERIAL PROPERTIES
# ------------------------------------------------------------
# Main advantage:
# Change concrete grade, steel grade or density here.
# The OpenSees model reads these values automatically.
# ------------------------------------------------------------

materials = pd.DataFrame([
    {
        "Material": "Concrete",
        "Grade": "C30/37",
        "E_MPa": 32800,
        "Poisson": 0.20,
        "Density_kN_m3": 25.0,
        "fck_MPa": 30.0,
        "fyk_MPa": np.nan
    },

    {
        "Material": "Reinforcement",
        "Grade": "B500B",
        "E_MPa": 200000,
        "Poisson": 0.30,
        "Density_kN_m3": 78.5,
        "fck_MPa": np.nan,
        "fyk_MPa": 500.0
    },

    {
        "Material": "Structural Steel",
        "Grade": "S355",
        "E_MPa": 210000,
        "Poisson": 0.30,
        "Density_kN_m3": 78.5,
        "fck_MPa": np.nan,
        "fyk_MPa": 355.0
    }
])

display(materials)


# ------------------------------------------------------------
# 5. SECTION PROPERTIES
# ------------------------------------------------------------
# RC columns shown on the drawing are 700 x 700 mm.
#
# Additional sections can be added to this table without
# changing the rest of the model.
# ------------------------------------------------------------

sections = pd.DataFrame([
    {
        "Section": "RC_700x700",
        "Type": "RC Column",
        "Material": "Concrete",
        "b_mm": 700,
        "h_mm": 700,
        "A_mm2": 700 * 700,
        "Ix_mm4": (700 * 700**3) / 12,
        "Iy_mm4": (700**3 * 700) / 12
    },

    {
        "Section": "RC_500x500",
        "Type": "RC Column",
        "Material": "Concrete",
        "b_mm": 500,
        "h_mm": 500,
        "A_mm2": 500 * 500,
        "Ix_mm4": (500 * 500**3) / 12,
        "Iy_mm4": (500**3 * 500) / 12
    },

    {
        "Section": "RC_Beam_400x700",
        "Type": "RC Beam",
        "Material": "Concrete",
        "b_mm": 400,
        "h_mm": 700,
        "A_mm2": 400 * 700,
        "Ix_mm4": (400 * 700**3) / 12,
        "Iy_mm4": (700 * 400**3) / 12
    }
])

display(sections)


# ------------------------------------------------------------
# 6. SLAB PROPERTIES
# ------------------------------------------------------------
# Flat slab thickness can be changed here.
# The drawing identifies the main system as a concrete flat slab.
# ------------------------------------------------------------

slab = {
    "type": "RC Flat Slab",
    "thickness_m": 0.300,
    "material": "Concrete",

    # Self-weight automatically calculated from thickness
    # and concrete density
    "include_self_weight": True,
}


# ------------------------------------------------------------
# 7. LOADS
# ------------------------------------------------------------
# THIS IS THE MAIN TABLE YOU WILL CHANGE WHEN TESTING LOADS.
#
# Gk = permanent load
# Qk = imposed load
# Wind loads can be added separately.
#
# Values are entered as area loads in kN/m².
# ------------------------------------------------------------

loads = pd.DataFrame([
    {
        "LoadCase": "Gk",
        "Description": "Slab self weight",
        "Load_kN_m2": 25.0 * slab["thickness_m"],
        "Direction": "Z"
    },

    {
        "LoadCase": "Gk",
        "Description": "Finishes + ceiling + services",
        "Load_kN_m2": 1.50,
        "Direction": "Z"
    },

    {
        "LoadCase": "Gk",
        "Description": "Partitions allowance",
        "Load_kN_m2": 1.00,
        "Direction": "Z"
    },

    {
        "LoadCase": "Qk",
        "Description": "Imposed load - general areas",
        "Load_kN_m2": 3.00,
        "Direction": "Z"
    },

    {
        "LoadCase": "Qk",
        "Description": "Plant / heavy areas",
        "Load_kN_m2": 5.00,
        "Direction": "Z"
    }
])

display(loads)


# ------------------------------------------------------------
# 8. WIND LOADS
# ------------------------------------------------------------
# Keep these separate so they can easily be changed later.
# Positive X/Y directions can be reversed by changing the sign.
# ------------------------------------------------------------

wind = pd.DataFrame([
    {
        "LoadCase": "W_X+",
        "Direction": "X",
        "Pressure_kN_m2": 1.00
    },

    {
        "LoadCase": "W_X-",
        "Direction": "X",
        "Pressure_kN_m2": -1.00
    },

    {
        "LoadCase": "W_Y+",
        "Direction": "Y",
        "Pressure_kN_m2": 1.00
    },

    {
        "LoadCase": "W_Y-",
        "Direction": "Y",
        "Pressure_kN_m2": -1.00
    }
])

display(wind)


# ------------------------------------------------------------
# 9. LOAD COMBINATIONS
# ------------------------------------------------------------
# These are example factors for analysis.
# They can be changed without touching the OpenSees code.
#
# For a final design, use the project/code-specific
# Eurocode combinations.
# ------------------------------------------------------------

combinations = pd.DataFrame([
    {
        "Combination": "ULS_01",
        "Gk": 1.35,
        "Qk": 1.50,
        "W_X+": 0.00,
        "W_X-": 0.00,
        "W_Y+": 0.00,
        "W_Y-": 0.00
    },

    {
        "Combination": "ULS_WX+",
        "Gk": 1.35,
        "Qk": 1.50,
        "W_X+": 1.50,
        "W_X-": 0.00,
        "W_Y+": 0.00,
        "W_Y-": 0.00
    },

    {
        "Combination": "ULS_WY+",
        "Gk": 1.35,
        "Qk": 1.50,
        "W_X+": 0.00,
        "W_X-": 0.00,
        "W_Y+": 1.50,
        "W_Y-": 0.00
    },

    {
        "Combination": "SLS",
        "Gk": 1.00,
        "Qk": 1.00,
        "W_X+": 0.00,
        "W_X-": 0.00,
        "W_Y+": 0.00,
        "W_Y-": 0.00
    }
])

display(combinations)


# ------------------------------------------------------------
# 10. DESIGN PARAMETERS
# ------------------------------------------------------------

DESIGN = {
    "gamma_M0": 1.00,
    "gamma_C": 1.50,
    "gamma_S": 1.15,

    # Concrete cover
    "cover_mm": 40,

    # Preliminary maximum reinforcement ratio
    "rho_max": 0.04,

    # Preliminary minimum reinforcement ratio
    "rho_min": 0.0013,
}


# ------------------------------------------------------------
# 11. MODEL SUMMARY
# ------------------------------------------------------------

print("=" * 60)
print("MODEL INPUT SUMMARY")
print("=" * 60)

print(f"Model:              {MODEL['name']}")
print(f"Storeys:            {MODEL['n_storeys']}")
print(f"Storey height:      {MODEL['storey_height']:.2f} m")
print(f"Slab thickness:     {slab['thickness_m']:.3f} m")
print(f"Concrete:            C30/37")
print(f"Reinforcement:       B500B")
print(f"Main columns:        700 x 700 mm")
print(f"X dimension:         {grid_x['X_m'].max():.2f} m")
print(f"Y dimension:         {grid_y['Y_m'].max():.2f} m")
print("=" * 60)

GRID X


,Grid,X_m
0,A,0.000
1,B,7.950
2,C,16.200
3,D,24.450
4,E,32.700
5,F,40.316
6,F',41.124
7,G,49.200
8,H,57.450
9,I,65.400



GRID Y


,Grid,Y_m
0,1,0.0
1,2,9.1
2,3,18.4
3,4,27.4
4,5,40.4
5,6,53.4



STOREY LEVELS


...

,Section,Type,Material,b_mm,h_mm,A_mm2,Ix_mm4,Iy_mm4
0,RC_700x700,RC Column,Concrete,700,700,490000,2.000833e+10,2.000833e+10
1,RC_500x500,RC Column,Concrete,500,500,250000,5.208333e+09,5.208333e+09
2,RC_Beam_400x700,RC Beam,Concrete,400,700,280000,1.143333e+10,3.733333e+09


,LoadCase,Description,Load_kN_m2,Direction
0,Gk,Slab self weight,7.5,Z
1,Gk,Finishes + ceiling + services,1.5,Z
2,Gk,Partitions allowance,1.0,Z
3,Qk,Imposed load - general areas,3.0,Z
4,Qk,Plant / heavy areas,5.0,Z


,LoadCase,Direction,Pressure_kN_m2
0,W_X+,X,1.0
1,W_X-,X,-1.0
2,W_Y+,Y,1.0
3,W_Y-,Y,-1.0


,Combination,Gk,Qk,W_X+,W_X-,W_Y+,W_Y-
0,ULS_01,1.35,1.5,0.0,0.0,0.0,0.0
1,ULS_WX+,1.35,1.5,1.5,0.0,0.0,0.0
2,ULS_WY+,1.35,1.5,0.0,0.0,1.5,0.0
3,SLS,1.00,1.0,0.0,0.0,0.0,0.0


MODEL INPUT SUMMARY
Model:              ATU 3D RC Structure
Storeys:            8
Storey height:      3.50 m
Slab thickness:     0.300 m
Concrete:            C30/37
Reinforcement:       B500B
Main columns:        700 x 700 mm
X dimension:         65.40 m
Y dimension:         53.40 m


In [4]:
# ============================================================
# 3D STRUCTURE VIEWER
# ============================================================
# Interactive Plotly viewer
#
# Requires:
#   - grid_x
#   - grid_y
#   - storeys
#   - sections
#   - loads
#   - OpenSees model already created and analysed
#
# Toggle:
#   Structure
#   Loads
#   Deformed shape
#   N, Vy, Vz, My, Mz, T
# ============================================================

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots


# ------------------------------------------------------------
# 1. GET NODE COORDINATES FROM OPENSEES
# ------------------------------------------------------------

def get_node_coordinates():

    node_data = []

    for node in ops.getNodeTags():

        coord = ops.nodeCoord(node)

        node_data.append({
            "Node": node,
            "X": coord[0],
            "Y": coord[1],
            "Z": coord[2]
        })

    return pd.DataFrame(node_data)


df_nodes = get_node_coordinates()


# ------------------------------------------------------------
# 2. GET ELEMENT CONNECTIVITY
# ------------------------------------------------------------

def get_element_connectivity():

    element_data = []

    for element in ops.getEleTags():

        nodes = ops.eleNodes(element)

        element_data.append({
            "Element": element,
            "NodeI": nodes[0],
            "NodeJ": nodes[1]
        })

    return pd.DataFrame(element_data)


df_elements = get_element_connectivity()


# ------------------------------------------------------------
# 3. GET ELEMENT INTERNAL FORCES
# ------------------------------------------------------------
#
# For a 3D elasticBeamColumn:
#
# [ P_i, Vy_i, Vz_i, T_i, My_i, Mz_i,
#   P_j, Vy_j, Vz_j, T_j, My_j, Mz_j ]
#
# The exact sign convention follows OpenSees local axes.
# ------------------------------------------------------------

def get_element_forces():

    results = []

    for element in ops.getEleTags():

        try:
            f = np.array(ops.eleResponse(element, "localForce"),
                         dtype=float)

            if len(f) >= 12:

                results.append({
                    "Element": element,

                    "N_i":  f[0],
                    "Vy_i": f[1],
                    "Vz_i": f[2],
                    "T_i":  f[3],
                    "My_i": f[4],
                    "Mz_i": f[5],

                    "N_j":  f[6],
                    "Vy_j": f[7],
                    "Vz_j": f[8],
                    "T_j":  f[9],
                    "My_j": f[10],
                    "Mz_j": f[11],
                })

        except Exception:
            pass

    return pd.DataFrame(results)


df_forces = get_element_forces()


# ------------------------------------------------------------
# 4. COMBINE ELEMENT + FORCE DATA
# ------------------------------------------------------------

df_results = df_elements.merge(
    df_forces,
    on="Element",
    how="left"
)

# Maximum absolute value in each element
df_results["N_max"] = df_results[["N_i", "N_j"]].abs().max(axis=1)
df_results["Vy_max"] = df_results[["Vy_i", "Vy_j"]].abs().max(axis=1)
df_results["Vz_max"] = df_results[["Vz_i", "Vz_j"]].abs().max(axis=1)
df_results["T_max"] = df_results[["T_i", "T_j"]].abs().max(axis=1)
df_results["My_max"] = df_results[["My_i", "My_j"]].abs().max(axis=1)
df_results["Mz_max"] = df_results[["Mz_i", "Mz_j"]].abs().max(axis=1)


# ------------------------------------------------------------
# 5. NODE DISPLACEMENTS
# ------------------------------------------------------------

def get_node_displacements():

    disp = []

    for node in ops.getNodeTags():

        try:
            d = ops.nodeDisp(node)

            disp.append({
                "Node": node,
                "UX": d[0],
                "UY": d[1],
                "UZ": d[2]
            })

        except Exception:

            disp.append({
                "Node": node,
                "UX": 0.0,
                "UY": 0.0,
                "UZ": 0.0
            })

    return pd.DataFrame(disp)


df_disp = get_node_displacements()


# ------------------------------------------------------------
# 6. DEFORMED COORDINATES
# ------------------------------------------------------------

df_plot_nodes = df_nodes.merge(
    df_disp,
    on="Node",
    how="left"
)

df_plot_nodes["Xd"] = (
    df_plot_nodes["X"]
    + df_plot_nodes["UX"] * MODEL["deformation_scale"]
)

df_plot_nodes["Yd"] = (
    df_plot_nodes["Y"]
    + df_plot_nodes["UY"] * MODEL["deformation_scale"]
)

df_plot_nodes["Zd"] = (
    df_plot_nodes["Z"]
    + df_plot_nodes["UZ"] * MODEL["deformation_scale"]
)


# ------------------------------------------------------------
# 7. LOAD VECTOR DATA
# ------------------------------------------------------------
#
# If your previous model has direct nodal loads, this section
# reads them from the "loads" input table when possible.
#
# For a future version, this can also display automatically
# generated tributary-area loads.
# ------------------------------------------------------------

load_vectors = []

# Look for a direct nodal load table if it exists
if "df_nodal_loads" in globals():

    for _, row in df_nodal_loads.iterrows():

        node = int(row["Node"])

        if node in df_nodes["Node"].values:

            xyz = df_nodes.loc[
                df_nodes["Node"] == node
            ].iloc[0]

            fx = row.get("Fx", 0.0)
            fy = row.get("Fy", 0.0)
            fz = row.get("Fz", 0.0)

            load_vectors.append({
                "Node": node,
                "X": xyz["X"],
                "Y": xyz["Y"],
                "Z": xyz["Z"],
                "Fx": fx,
                "Fy": fy,
                "Fz": fz
            })


df_loads_plot = pd.DataFrame(load_vectors)


# ------------------------------------------------------------
# 8. FORCE SELECTION
# ------------------------------------------------------------

FORCE_OPTIONS = {
    "None": None,

    "Axial N": "N_max",
    "Shear Vy": "Vy_max",
    "Shear Vz": "Vz_max",

    "Moment My": "My_max",
    "Moment Mz": "Mz_max",

    "Torsion T": "T_max"
}


# ------------------------------------------------------------
# 9. CREATE STRUCTURE LINES
# ------------------------------------------------------------

def create_structure_trace():

    x = []
    y = []
    z = []

    for _, element in df_elements.iterrows():

        ni = element["NodeI"]
        nj = element["NodeJ"]

        pi = df_nodes[df_nodes["Node"] == ni].iloc[0]
        pj = df_nodes[df_nodes["Node"] == nj].iloc[0]

        x += [pi["X"], pj["X"], None]
        y += [pi["Y"], pj["Y"], None]
        z += [pi["Z"], pj["Z"], None]

    return go.Scatter3d(
        x=x,
        y=y,
        z=z,
        mode="lines",
        name="Structure",
        line=dict(
            width=5
        ),
        hoverinfo="none"
    )


# ------------------------------------------------------------
# 10. CREATE FORCE-COLOURED MEMBERS
# ------------------------------------------------------------

def create_force_trace(force_name):

    force_column = FORCE_OPTIONS[force_name]

    if force_column is None:
        return None

    x = []
    y = []
    z = []
    values = []

    for _, element in df_results.iterrows():

        ni = element["NodeI"]
        nj = element["NodeJ"]

        pi = df_nodes[df_nodes["Node"] == ni].iloc[0]
        pj = df_nodes[df_nodes["Node"] == nj].iloc[0]

        value = element[force_column]

        x += [pi["X"], pj["X"], None]
        y += [pi["Y"], pj["Y"], None]
        z += [pi["Z"], pj["Z"], None]

        values += [value, value, None]

    return go.Scatter3d(
        x=x,
        y=y,
        z=z,

        mode="lines",

        name=force_name,

        line=dict(
            width=8,
            color=values,
            colorscale="RdBu",
            reversescale=True,
            colorbar=dict(
                title=force_name
            ),
            cmin=-max(
                abs(df_results[force_column].min()),
                abs(df_results[force_column].max())
            ),
            cmax=max(
                abs(df_results[force_column].min()),
                abs(df_results[force_column].max())
            )
        ),

        hovertemplate=(
            "Element: %{customdata[0]}<br>"
            + force_name
            + ": %{customdata[1]:.2f}<extra></extra>"
        ),

        customdata=[
            [element["Element"], element[force_column]]
            if value is not None else
            [None, None]

            for element, value
            in zip(
                df_results.to_dict("records"),
                values
            )
        ]
    )


# ------------------------------------------------------------
# 11. CREATE LOAD ARROWS
# ------------------------------------------------------------

def create_load_trace():

    if df_loads_plot.empty:
        return None

    # Automatic visual scaling
    max_load = max(
        df_loads_plot[
            ["Fx", "Fy", "Fz"]
        ].abs().max().max(),
        1.0
    )

    model_size = max(
        df_nodes["X"].max() - df_nodes["X"].min(),
        df_nodes["Y"].max() - df_nodes["Y"].min(),
        df_nodes["Z"].max() - df_nodes["Z"].min()
    )

    arrow_scale = model_size * 0.12 / max_load

    x = []
    y = []
    z = []

    for _, row in df_loads_plot.iterrows():

        x += [
            row["X"],
            row["X"] + row["Fx"] * arrow_scale,
            None
        ]

        y += [
            row["Y"],
            row["Y"] + row["Fy"] * arrow_scale,
            None
        ]

        z += [
            row["Z"],
            row["Z"] + row["Fz"] * arrow_scale,
            None
        ]

    return go.Scatter3d(
        x=x,
        y=y,
        z=z,

        mode="lines",

        name="Loads",

        line=dict(
            width=6
        ),

        hoverinfo="none"
    )


# ------------------------------------------------------------
# 12. CREATE DEFORMED STRUCTURE
# ------------------------------------------------------------

def create_deformed_trace():

    x = []
    y = []
    z = []

    for _, element in df_elements.iterrows():

        ni = element["NodeI"]
        nj = element["NodeJ"]

        pi = df_plot_nodes[
            df_plot_nodes["Node"] == ni
        ].iloc[0]

        pj = df_plot_nodes[
            df_plot_nodes["Node"] == nj
        ].iloc[0]

        x += [pi["Xd"], pj["Xd"], None]
        y += [pi["Yd"], pj["Yd"], None]
        z += [pi["Zd"], pj["Zd"], None]

    return go.Scatter3d(
        x=x,
        y=y,
        z=z,

        mode="lines",

        name="Deformed shape",

        line=dict(
            width=4,
            dash="dash"
        )
    )


# ------------------------------------------------------------
# 13. STRUCTURE NODES
# ------------------------------------------------------------

def create_nodes_trace():

    return go.Scatter3d(
        x=df_nodes["X"],
        y=df_nodes["Y"],
        z=df_nodes["Z"],

        mode="markers",

        name="Nodes",

        marker=dict(
            size=3
        ),

        text=[
            f"Node {n}"
            for n in df_nodes["Node"]
        ],

        hovertemplate="%{text}<extra></extra>"
    )


# ------------------------------------------------------------
# 14. MAIN FIGURE
# ------------------------------------------------------------

def plot_structure(
    show_loads=False,
    show_deformed=False,
    force="None",
    show_nodes=False
):

    fig = go.Figure()

    # --------------------------------------------------------
    # STRUCTURE
    # --------------------------------------------------------

    force_trace = create_force_trace(force)

    if force_trace is not None:
        fig.add_trace(force_trace)

    else:
        fig.add_trace(
            create_structure_trace()
        )

    # --------------------------------------------------------
    # NODES
    # --------------------------------------------------------

    if show_nodes:
        fig.add_trace(
            create_nodes_trace()
        )

    # --------------------------------------------------------
    # LOADS
    # --------------------------------------------------------

    if show_loads:

        load_trace = create_load_trace()

        if load_trace is not None:
            fig.add_trace(load_trace)

    # --------------------------------------------------------
    # DEFORMED SHAPE
    # --------------------------------------------------------

    if show_deformed:

        fig.add_trace(
            create_deformed_trace()
        )

    # --------------------------------------------------------
    # AXIS / LAYOUT
    # --------------------------------------------------------

    fig.update_layout(

        title={
            "text": (
                "3D Structural Model"
                if force == "None"
                else f"3D Internal Force View — {force}"
            ),
            "x": 0.5
        },

        scene=dict(

            xaxis_title="X [m]",
            yaxis_title="Y [m]",
            zaxis_title="Z [m]",

            aspectmode="data",

            camera=dict(
                eye=dict(
                    x=1.6,
                    y=1.6,
                    z=1.2
                )
            )
        ),

        height=750,

        legend=dict(
            x=0.01,
            y=0.99
        )
    )

    fig.show()


# ------------------------------------------------------------
# 15. EXAMPLE VIEWS
# ------------------------------------------------------------

# Basic structure
plot_structure()

# Uncomment to show loads:
# plot_structure(show_loads=True)

# Uncomment for axial force:
# plot_structure(force="Axial N")

# Uncomment for bending moment:
# plot_structure(force="Moment Mz")

# Uncomment for deformed shape:
# plot_structure(show_deformed=True)